# Fraud Shield - Hybrid Ensemble

Combines the supervised branch (`03_baseline_models.ipynb`: LogReg, RF,
XGBoost) and the deep learning branch (`04_deep_learning_models.ipynb`:
FNN, LSTM) into a single interpretable meta-learner, then evaluates the
full hybrid system against the true held-out `fraudTest.csv` -- this is
the number that goes in the final performance report.

**Inputs:**
- `data/processed/models/baseline_val_predictions.parquet`
- `data/processed/models/dl_val_predictions.parquet`
- `data/processed/models/*.joblib`, `*.pt`, `feature_scaler.joblib`
- `data/processed/test_features.parquet` (the true held-out set)

**Outputs:** `meta_model.joblib`, final metrics on the held-out test set


In [ ]:
import sys
sys.path.append('..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from sklearn.model_selection import train_test_split

from src.models.ensemble import build_meta_features, train_meta_model, predict as meta_predict
from src.models.evaluate import compute_metrics, print_report
from src.models.deep_learning import FraudFNN, FraudLSTM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = Path('../data/processed/models')


## 1. Load validation predictions from both branches

Both files were produced from the *same* validation split
(`random_state=42`, `test_size=0.2` on `train_features.parquet`), so
their rows line up directly -- verified below rather than assumed.


In [ ]:
baseline_val = pd.read_parquet(MODELS_DIR / 'baseline_val_predictions.parquet')
dl_val = pd.read_parquet(MODELS_DIR / 'dl_val_predictions.parquet')

assert len(baseline_val) == len(dl_val), 'Row count mismatch between branches'
assert np.array_equal(baseline_val['is_fraud'].values, dl_val['is_fraud'].values), (
    'Row order mismatch -- baseline and DL notebooks must use the same split'
)
print(f'{len(baseline_val)} aligned validation rows across both branches.')


## 2. Split the validation set into meta-train / meta-val

Fitting *and* evaluating the meta-learner on the same rows the base models
were validated on would overstate performance. Splitting again here gives
the meta-learner its own held-out slice, independent of what the base
models saw during their own validation.


In [ ]:
y_meta = baseline_val['is_fraud'].values

meta_features = build_meta_features(
    baseline_val['logreg_proba'].values,
    baseline_val['rf_proba'].values,
    baseline_val['xgb_proba'].values,
    dl_val['fnn_proba'].values,
    dl_val['lstm_proba'].values,
)

meta_X_train, meta_X_val, meta_y_train, meta_y_val = train_test_split(
    meta_features, y_meta, test_size=0.3, stratify=y_meta, random_state=42
)

print(f'Meta-train: {meta_X_train.shape}, meta-val: {meta_X_val.shape}')


## 3. Train the meta-model

A simple logistic regression over the five base-model probabilities --
kept deliberately interpretable (coefficients show how much the ensemble
trusts each branch) rather than another black-box model on top of
black-box models.


In [ ]:
meta_model = train_meta_model(meta_X_train, meta_y_train)

base_model_names = ['logreg', 'random_forest', 'xgboost', 'fnn', 'lstm']
coef_df = pd.Series(meta_model.coef_[0], index=base_model_names).sort_values(ascending=False)
print('Meta-model coefficients (higher = more trusted by the ensemble):')
print(coef_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
coef_df.plot(kind='barh', ax=ax, color='slateblue')
ax.set_title('Hybrid ensemble - how much each base model is weighted')
plt.tight_layout()
plt.show()


## 4. Evaluate the ensemble on meta-val

Compares the hybrid ensemble against each individual base model on the
same rows, to check the ensemble is actually adding value rather than
just matching the best single model.


In [ ]:
ensemble_proba = meta_predict(
    meta_model,
    meta_X_val[:, 0], meta_X_val[:, 1], meta_X_val[:, 2], meta_X_val[:, 3], meta_X_val[:, 4],
)
ensemble_pred = (ensemble_proba >= 0.5).astype(int)

ensemble_metrics = compute_metrics(meta_y_val, ensemble_pred, ensemble_proba)

individual_metrics = {}
for i, name in enumerate(base_model_names):
    proba_i = meta_X_val[:, i]
    pred_i = (proba_i >= 0.5).astype(int)
    individual_metrics[name] = compute_metrics(meta_y_val, pred_i, proba_i)

comparison = pd.DataFrame({**individual_metrics, 'hybrid_ensemble': ensemble_metrics}).T
comparison = comparison.sort_values('f1', ascending=False)
comparison


In [ ]:
print_report(meta_y_val, ensemble_pred)


## 5. Save the meta-model


In [ ]:
joblib.dump(meta_model, MODELS_DIR / 'meta_model.joblib')
print('Saved meta_model.joblib to', MODELS_DIR)


## 6. Final evaluation on the true held-out test set

Everything up to this point used validation splits carved out of
`fraudTrain.csv`. This section runs the full hybrid pipeline -- all five
base models plus the meta-model -- against `fraudTest.csv`, which no model
has seen in any form. This is the number for the performance report.


In [ ]:
test_df = pd.read_parquet('../data/processed/test_features.parquet')

id_cols = ['transaction_id', 'event_time', 'cc_num']
target_col = 'is_fraud'
feature_cols = [c for c in test_df.columns if c not in id_cols + [target_col]]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f'Held-out test shape: {X_test.shape}, fraud rate {y_test.mean()*100:.4f}%')


### 6a. Supervised branch predictions on test


In [ ]:
logreg_model = joblib.load(MODELS_DIR / 'logreg.joblib')
rf_model = joblib.load(MODELS_DIR / 'random_forest.joblib')
xgb_model = joblib.load(MODELS_DIR / 'xgboost.joblib')

test_logreg_proba = logreg_model.predict_proba(X_test)[:, 1]
test_rf_proba = rf_model.predict_proba(X_test)[:, 1]
test_xgb_proba = xgb_model.predict_proba(X_test)[:, 1]


### 6b. Deep learning branch predictions on test

Rebuilds sequences for the test set using the same
`groupby().shift()` approach from `04_deep_learning_models.ipynb`.


In [ ]:
scaler = joblib.load(MODELS_DIR / 'feature_scaler.joblib')
SEQ_LEN = 5

def build_sequences(df, feature_cols, seq_len):
    sorted_df = df.sort_values(['cc_num', 'event_time']).reset_index()
    sorted_df = sorted_df.rename(columns={'index': 'orig_index'})
    lag_arrays = [sorted_df[feature_cols].values]
    for lag in range(1, seq_len):
        shifted = sorted_df.groupby('cc_num')[feature_cols].shift(lag).fillna(0).values
        lag_arrays.append(shifted)
    lag_arrays = lag_arrays[::-1]
    sequences = np.stack(lag_arrays, axis=1)
    pos_lookup = pd.Series(np.arange(len(sorted_df)), index=sorted_df['orig_index'])
    return sequences, pos_lookup

test_sequences, test_pos_lookup = build_sequences(test_df, feature_cols, SEQ_LEN)
test_positions = test_pos_lookup.loc[X_test.index].values
X_test_seq_raw = test_sequences[test_positions]

assert np.array_equal(test_df.iloc[test_positions][target_col].values, y_test.values)

n_test, seq_len, n_features = X_test_seq_raw.shape
X_test_scaled = scaler.transform(X_test.values)
X_test_seq_scaled = scaler.transform(X_test_seq_raw.reshape(-1, n_features)).reshape(n_test, seq_len, n_features)


In [ ]:
fnn = FraudFNN(input_dim=X_test_scaled.shape[1]).to(device)
fnn.load_state_dict(torch.load(MODELS_DIR / 'fnn.pt', map_location=device))
fnn.eval()

lstm = FraudLSTM(input_dim=n_features).to(device)
lstm.load_state_dict(torch.load(MODELS_DIR / 'lstm.pt', map_location=device))
lstm.eval()

with torch.no_grad():
    test_fnn_proba = fnn(torch.tensor(X_test_scaled, dtype=torch.float32).to(device)).cpu().numpy()
    test_lstm_proba = lstm(torch.tensor(X_test_seq_scaled, dtype=torch.float32).to(device)).cpu().numpy()


### 6c. Final hybrid ensemble prediction on test


In [ ]:
test_meta_features = build_meta_features(
    test_logreg_proba, test_rf_proba, test_xgb_proba, test_fnn_proba, test_lstm_proba
)
test_ensemble_proba = meta_model.predict_proba(test_meta_features)[:, 1]
test_ensemble_pred = (test_ensemble_proba >= 0.5).astype(int)

final_metrics = compute_metrics(y_test, test_ensemble_pred, test_ensemble_proba)
print('Final hybrid ensemble - held-out test performance:')
print(final_metrics)
print()
print_report(y_test, test_ensemble_pred)


### 6d. Full comparison table (held-out test)

The table to put in the final performance report -- every model, on the
one dataset none of them ever saw.


In [ ]:
test_individual = {
    'logreg': compute_metrics(y_test, (test_logreg_proba >= 0.5).astype(int), test_logreg_proba),
    'random_forest': compute_metrics(y_test, (test_rf_proba >= 0.5).astype(int), test_rf_proba),
    'xgboost': compute_metrics(y_test, (test_xgb_proba >= 0.5).astype(int), test_xgb_proba),
    'fnn': compute_metrics(y_test, (test_fnn_proba >= 0.5).astype(int), test_fnn_proba),
    'lstm': compute_metrics(y_test, (test_lstm_proba >= 0.5).astype(int), test_lstm_proba),
}

final_comparison = pd.DataFrame({**test_individual, 'hybrid_ensemble': final_metrics}).T
final_comparison = final_comparison.sort_values('f1', ascending=False)
final_comparison


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
final_comparison[['precision', 'recall', 'f1', 'auc_roc']].plot(kind='bar', ax=ax)
ax.set_title('Final held-out test performance - all models')
ax.legend(loc='lower right')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Save results for the performance report


In [ ]:
final_comparison.to_csv('../docs/final_test_results.csv')
print('Saved final_test_results.csv to docs/ -- copy into docs/performance_report.md')


## 8. Summary

Fill in once run against the real data:

- Does the hybrid ensemble outperform the best single model (likely
  XGBoost) on F1 / AUC-ROC? By how much?
- Which base model does the meta-learner trust most (see the coefficient
  plot in section 3)?
- Any notable precision/recall trade-off worth calling out for the
  performance report?

**Next:** wire `meta_model.joblib` plus the five base models into
`src/pipeline/pipeline_definition.py` and `app/inference_client.py` so the
same ensemble logic runs behind the SageMaker endpoint and the Streamlit
app, not just in this notebook.
